In [0]:
import pandas as pd
import logging
from fpdf import FPDF
from datetime import datetime
from typing import Dict, Any, Optional

# Configuración de logging
logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(levelname)s - %(message)s')
logger = logging.getLogger(__name__)

class ReportPDF(FPDF):
    """Clase para la generación del documento PDF del reporte ROS."""
    def header(self) -> None:
        self.set_font('Arial', 'B', 15)
        self.cell(0, 10, 'INFORME DE INTELIGENCIA FINANCIERA (ROS)', 0, 1, 'C')
        self.ln(5)

    def footer(self) -> None:
        self.set_y(-15)
        self.set_font('Arial', 'I', 8)
        self.cell(0, 10, f'Página {self.page_no()}', 0, 0, 'C')

def fetch_summary_metrics(spark_session: Any) -> Dict[str, int]:
    """
    Obtiene las métricas totales y conteo de alertas desde las tablas Gold y Silver.
    """
    logger.info("Consultando métricas de resumen en Databricks...")
    query = """
    SELECT 
        (SELECT COUNT(*) FROM workspace.aml_proyect.transacciones_plata_robert) as total_tx,
        (SELECT COUNT(*) FROM workspace.aml_proyect.gold_perfiles_riesgo_cliente) as total_clientes,
        (SELECT SUM(Flag_Structuring_Pitufeo) FROM workspace.aml_proyect.gold_perfiles_riesgo_cliente) as alertas_structuring,
        (SELECT SUM(Flag_Fan_In_Plataforma) FROM workspace.aml_proyect.gold_perfiles_riesgo_cliente) as alertas_fanin,
        (SELECT SUM(Flag_Crypto_Mixer) FROM workspace.aml_proyect.gold_perfiles_riesgo_cliente) as alertas_crypto
    """
    try:
        df_spark = spark_session.sql(query)
        df_pandas = df_spark.toPandas()
        
        metrics = {
            'total_tx': int(df_pandas.iloc[0]['total_tx'] or 0),
            'total_clientes': int(df_pandas.iloc[0]['total_clientes'] or 0),
            'alertas_structuring': int(df_pandas.iloc[0]['alertas_structuring'] or 0),
            'alertas_fanin': int(df_pandas.iloc[0]['alertas_fanin'] or 0),
            'alertas_crypto': int(df_pandas.iloc[0]['alertas_crypto'] or 0)
        }
        logger.info("Métricas de resumen obtenidas correctamente.")
        return metrics
    except Exception as e:
        logger.error(f"Fallo al consultar las métricas de resumen: {e}")
        raise

def fetch_top_subjects(spark_session: Any) -> Dict[str, pd.DataFrame]:
    """
    Obtiene los sujetos de interés con mayor riesgo por cada tipología.
    """
    logger.info("Consultando sujetos de interés críticos...")
    queries = {
        'fanin': """
            SELECT PK_Cliente, Total_Tx_Recibidas, Monto_Total_Enviado_USD, Ratio_Operado_vs_Ingreso_Declarado 
            FROM workspace.aml_proyect.gold_perfiles_riesgo_cliente 
            WHERE Flag_Fan_In_Plataforma = 1 
            ORDER BY Total_Movido DESC LIMIT 1
        """,
        'crypto': """
            SELECT PK_Cliente, Monto_Total_Enviado_USD, Ratio_Tx_Crypto, Tx_Canal_Crypto 
            FROM workspace.aml_proyect.gold_perfiles_riesgo_cliente 
            WHERE Flag_Crypto_Mixer = 1 
            ORDER BY Total_Movido DESC LIMIT 1
        """,
        'structuring': """
            SELECT PK_Cliente, Total_Tx_Enviadas, Monto_Total_Enviado_USD 
            FROM workspace.aml_proyect.gold_perfiles_riesgo_cliente 
            WHERE Flag_Structuring_Pitufeo = 1 
            ORDER BY Total_Movido DESC LIMIT 1
        """
    }
    
    subjects = {}
    for tipologia, query in queries.items():
        try:
            subjects[tipologia] = spark_session.sql(query).toPandas()
        except Exception as e:
            logger.error(f"Fallo al consultar sujeto crítico para tipología {tipologia}: {e}")
            raise
    
    logger.info("Sujetos críticos obtenidos correctamente.")
    return subjects

def generate_pdf_report(metrics: Dict[str, int], subjects: Dict[str, pd.DataFrame], output_path: str) -> None:
    """
    Genera y guarda el documento PDF con el Informe de Inteligencia Financiera.
    """
    logger.info("Iniciando la generación del documento PDF...")
    try:
        pdf = ReportPDF()
        pdf.add_page()

        pdf.set_font("Arial", 'B', 12)
        pdf.cell(0, 8, "A: Comité de Cumplimiento Normativo / Oficial de Cumplimiento", ln=1)
        pdf.cell(0, 8, "DE: Unidad de Inteligencia Financiera (UIF) - Análisis Transaccional", ln=1)
        pdf.cell(0, 8, f"FECHA: {datetime.now().strftime('%d/%m/%Y')}", ln=1)
        pdf.cell(0, 8, "CLASIFICACIÓN: CONFIDENCIAL / RESTRINGIDO", ln=1)
        pdf.ln(5)

        total_alertas = metrics['alertas_structuring'] + metrics['alertas_fanin'] + metrics['alertas_crypto']

        pdf.set_font("Arial", 'B', 14)
        pdf.cell(0, 10, "1. Resumen de Hallazgos (Data Oficial Lakehouse)", ln=1)
        pdf.set_font("Arial", '', 11)
        resumen = f"En el último barrido de monitoreo transaccional se analizó un universo de {metrics['total_tx']:,} operaciones financieras consolidadas, correspondientes a una cartera de {metrics['total_clientes']:,} clientes activos. Tras aplicar la matriz de riesgo y reglas heurísticas de detección en Databricks, el sistema ha emitido {total_alertas:,} alertas confirmadas de alto riesgo."
        pdf.multi_cell(0, 8, resumen.encode('latin-1', 'replace').decode('latin-1'))
        pdf.ln(3)

        pdf.cell(0, 8, "Distribución de alertas por tipología:", ln=1)
        pdf.cell(0, 8, f"- Concentración de Fondos (Fan-In): {metrics['alertas_fanin']} alertas.", ln=1)
        pdf.cell(0, 8, f"- Ofuscación con Criptoactivos (Crypto Mixers): {metrics['alertas_crypto']} alertas.", ln=1)
        pdf.cell(0, 8, f"- Estructuración (Pitufeo/Structuring): {metrics['alertas_structuring']} alertas.", ln=1)
        pdf.ln(5)

        pdf.set_font("Arial", 'B', 14)
        pdf.cell(0, 10, "2. Análisis de Patrones y Comportamiento", ln=1)
        pdf.set_font("Arial", '', 11)
        patrones = "Fan-In: Clientes reciben ráfagas de micro-depósitos de múltiples contrapartes en lapsos muy cortos, para luego evacuar el capital de una sola vez.\nCrypto Mixers: Fondos ingresados son canalizados rápidamente al 100% hacia Exchanges de criptoactivos en transferencias de alta frecuencia, ocultando su trazabilidad.\nEstructuración: Depósitos en efectivo concentrados sistemáticamente justo por debajo del límite regulatorio de USD 10,000 para eludir controles."
        pdf.multi_cell(0, 8, patrones.encode('latin-1', 'replace').decode('latin-1'))
        pdf.ln(5)

        pdf.set_font("Arial", 'B', 14)
        pdf.cell(0, 10, "3. Sujetos de Interés (Casos Críticos Detectados)", ln=1)
        pdf.set_font("Arial", '', 11)

        sujeto_fanin = subjects.get('fanin')
        if sujeto_fanin is not None and not sujeto_fanin.empty:
            sf = sujeto_fanin.iloc[0]
            pdf.set_font("Arial", 'B', 11)
            pdf.cell(0, 8, f"- Sujeto ID: {sf['PK_Cliente']} (Fan-In)", ln=1)
            pdf.set_font("Arial", '', 11)
            obs = f"Recibió {sf['Total_Tx_Recibidas']:.0f} transferencias, luego envió un total de USD {sf['Monto_Total_Enviado_USD']:,.2f} con una relación de ingresos/operado de {sf['Ratio_Operado_vs_Ingreso_Declarado']:.2f}."
            pdf.multi_cell(0, 8, obs.encode('latin-1', 'replace').decode('latin-1'))
            
        sujeto_crypto = subjects.get('crypto')
        if sujeto_crypto is not None and not sujeto_crypto.empty:
            sc = sujeto_crypto.iloc[0]
            pdf.set_font("Arial", 'B', 11)
            pdf.cell(0, 8, f"- Sujeto ID: {sc['PK_Cliente']} (Crypto Mixer)", ln=1)
            pdf.set_font("Arial", '', 11)
            obs = f"Canalizó USD {sc['Monto_Total_Enviado_USD']:,.2f} a exchanges, representando el {sc['Ratio_Tx_Crypto']*100:.1f}% de sus transferencias de salida en {sc['Tx_Canal_Crypto']:.0f} envíos."
            pdf.multi_cell(0, 8, obs.encode('latin-1', 'replace').decode('latin-1'))

        sujeto_structuring = subjects.get('structuring')
        if sujeto_structuring is not None and not sujeto_structuring.empty:
            ss = sujeto_structuring.iloc[0]
            pdf.set_font("Arial", 'B', 11)
            pdf.cell(0, 8, f"- Sujeto ID: {ss['PK_Cliente']} (Estructuración)", ln=1)
            pdf.set_font("Arial", '', 11)
            obs = f"Generó {ss['Total_Tx_Enviadas']:.0f} transacciones de salida totalizando USD {ss['Monto_Total_Enviado_USD']:,.2f}, todas por montos menores a USD 3,000."
            pdf.multi_cell(0, 8, obs.encode('latin-1', 'replace').decode('latin-1'))

        pdf.ln(5)
        pdf.set_font("Arial", 'B', 14)
        pdf.cell(0, 10, "4. Plan de Acción Recomendado", ln=1)
        pdf.set_font("Arial", '', 11)
        acciones = "1. Bloqueo Preventivo Inmediato: Congelar transitoriamente las cuentas de los Sujetos de Interés.\n2. Ampliación de Debida Diligencia (EDD): Solicitar justificación comercial o de origen de fondos en efectivo y transferencias masivas.\n3. Reporte de Operaciones Sospechosas (ROS): Enviar a la UIF nacional adjuntando evidencia transaccional.\n4. Watchlists: Agregar cuentas contrapartes vinculadas a bloqueo preventivo."
        pdf.multi_cell(0, 8, acciones.encode('latin-1', 'replace').decode('latin-1'))

        pdf.output(output_path)
        logger.info(f"PDF generado exitosamente en: {output_path}")
    except Exception as e:
        logger.error(f"Fallo al generar el PDF: {e}")
        raise

def main() -> None:
    """
    Función principal que orquesta la generación del reporte en Databricks.
    """
    logger.info("Iniciando proceso de generación de reporte ROS.")
    
    try:
        # En Databricks, `spark` es provisto globalmente
        global_spark = spark  # noqa: F821
    except NameError:
        logger.error("La variable 'spark' no está definida. Este script debe ejecutarse en un entorno de Databricks.")
        return

    try:
        # 1. Obtener métricas
        metrics = fetch_summary_metrics(global_spark)
        
        # 2. Obtener sujetos críticos
        subjects = fetch_top_subjects(global_spark)
        
        # 3. Generar PDF
        output_path = "/databricks/driver/Reporte_ROS_Inteligencia_Financiera.pdf"
        generate_pdf_report(metrics, subjects, output_path)
        logger.info("Proceso de generación de reporte finalizado con éxito.")
    except Exception as e:
        logger.error(f"Proceso abortado debido a un error: {e}")

if __name__ == "__main__":
    main()
